In [19]:
!pip install icrawler

import os
from icrawler.builtin import BingImageCrawler
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Avengers list
avengers = [
    "Chris Evans face",
    "Chris Hemsworth face",
    "Mark Ruffalo face",
    "Robert Downey Jr face",
    "Scarlett Johansson face"
]

# Create dataset folder
DATASET_PATH = "dataset"
os.makedirs(DATASET_PATH, exist_ok=True)

# Download images
for person in avengers:
    folder_name = person.split()[0] + "_" + person.split()[1]
    person_path = os.path.join(DATASET_PATH, folder_name)

    os.makedirs(person_path, exist_ok=True)

    crawler = BingImageCrawler(storage={'root_dir': person_path})
    crawler.crawl(keyword=person, max_num=30)

print("Images Downloaded!")

# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

data = []
labels = []
IMG_SIZE = 100

# Process images
for person in os.listdir(DATASET_PATH):
    person_path = os.path.join(DATASET_PATH, person)

    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)

        try:
            img = cv2.imread(img_path)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            faces = face_cascade.detectMultiScale(gray, 1.3, 5)

            for (x, y, w, h) in faces:
                face = gray[y:y+h, x:x+w]
                face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))

                data.append(face.flatten())
                labels.append(person)

        except:
            pass

data = np.array(data)
labels = np.array(labels)

print("Dataset Created:", data.shape)

# Encode labels
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    data, labels_encoded, test_size=0.2, random_state=42
)

# Train model
model = SVC(kernel='linear')
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

# Save model
import pickle
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(le, open("labels.pkl", "wb"))

print("Model Saved Successfully!")

ERROR:downloader:Response status code 403, file https://wallpapercrafter.com/desktop1/614492-chris-hemsworth-celebrities-star-movie-actor-short.jpg
ERROR:downloader:Response status code 403, file https://images.wallpapersden.com/image/download/chris-hemsworth-actor-face_amdnaGaUmZqaraWkpJRma2ttrWdobW0.jpg
ERROR:downloader:Response status code 403, file https://www.thefashionisto.com/wp-content/uploads/2024/10/Chris-Hemsworth-Textured-Crop-Haircut-Heart-Shaped-Face.jpg
ERROR:downloader:Response status code 403, file https://images.wallpapersden.com/image/download/chris-hemsworth-actor-face_amdnaGaUmZqaraWkpJRmZmxrrWdpZWU.jpg
ERROR:downloader:Response status code 403, file https://p4.wallpaperbetter.com/wallpaper/762/936/854/chris-hemsworth-celebrities-star-movie-actor-short-hair-face-blue-eyes-chris-hemsworth-wallpaper-preview.jpg
ERROR:downloader:Response status code 403, file https://p4.wallpaperbetter.com/wallpaper/118/329/130/chris-hemsworth-beard-boy-face-wallpaper-preview.jpg
ERRO

Images Downloaded!
Dataset Created: (143, 10000)
Accuracy: 0.7586206896551724
Model Saved Successfully!
